# SafeScan - Step 2: Feature Extraction + Model Training (IIT Patna Dataset)

This notebook:
1. Re-loads your cropped images + OCR text (from Step 1 v2) - re-upload both zips
2. Cleans the noisy OCR text
3. Extracts ResNet50 image features + TF-IDF text features
4. Fuses both and trains the classifier
5. Saves the trained model + preprocessors for later use in the backend

**Runtime setup:** Runtime -> Change runtime type -> GPU (T4)


## Cell 1 - Load your cropped images and OCR text from Google Drive

Browser-based upload (`files.upload()`) is unreliable for large files (250MB+) and can hang
for a long time. Instead: upload both zips to your Google Drive first (drive.google.com,
drag and drop into a folder - e.g. a folder called `SafeScan`), then run the cell below to
mount Drive and copy the files directly - this is much faster since it's a server-to-server
copy instead of a browser upload.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Update this path to match where you put the files in your Drive
DRIVE_FOLDER = '/content/drive/MyDrive/SafeScan'

!cp "{DRIVE_FOLDER}/iitpatna_cropped.zip" /content/
!cp "{DRIVE_FOLDER}/iitpatna_ocr_text.zip" /content/

!unzip -q /content/iitpatna_cropped.zip -d /
!unzip -q /content/iitpatna_ocr_text.zip -d /

print("Cropped images ready:", os.listdir('/content/iitpatna_cropped')[:5])
print("OCR text ready:", os.listdir('/content/iitpatna_ocr_text')[:5])


: 

**If you get a `FileNotFoundError` after this** (same issue as last time - zips created
from an absolute path can nest an extra `/content/content/...` folder), run this fix:
```python
import shutil
if os.path.exists('/content/content'):
    shutil.move('/content/content/iitpatna_cropped', '/content/iitpatna_cropped')
    shutil.move('/content/content/iitpatna_ocr_text', '/content/iitpatna_ocr_text')
    shutil.rmtree('/content/content', ignore_errors=True)
print(os.listdir('/content/iitpatna_cropped')[:5])
```


## Cell 2 - Install/import required packages

In [ ]:
import json
import re
import numpy as np
import pickle
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.utils import to_categorical

print("All packages loaded.")


## Cell 3 - Text cleaning function (handles noisy OCR)

In [ ]:
def clean_ocr_text(raw_tokens):
    """
    Cleans a list of OCR-detected text tokens:
    - Lowercases everything
    - Removes tokens that are pure punctuation/symbols
    - Removes tokens shorter than 2 characters (likely OCR noise)
    - Joins into a single string
    """
    cleaned = []
    for tok in raw_tokens:
        tok = tok.strip().lower()
        tok = re.sub(r'[^a-z0-9\s\-]', '', tok)
        if len(tok) >= 2:
            cleaned.append(tok)
    return ' '.join(cleaned)

# quick test using a real noisy sample we saw earlier
print(clean_ocr_text(['"Zp lock', 'Jabsons', 'in', 'Traditions', 'Rich !', 'Mini',
                       'Bhakharwadi', 'famous', 'Maharashtras', "'Snack", 'Spicy ', 'Sweet ']))


## Cell 4 - Load images + cleaned OCR text together

In [ ]:
IMAGES_ROOT = '/content/iitpatna_cropped'
TEXT_ROOT = '/content/iitpatna_ocr_text'

def load_multimodal_data(images_root, text_root, max_per_class=None):
    """
    Loads image features (via ResNet50) and cleaned OCR text for every sample.
    Set max_per_class to an integer to limit samples per category for faster testing.
    """
    image_features = []
    text_data = []
    labels = []

    print("Loading ResNet50 (pretrained, frozen)...")
    model_cnn = ResNet50(weights='imagenet', include_top=False, pooling='avg')

    categories = sorted(os.listdir(images_root))
    print(f"Found {len(categories)} categories: {categories}")

    for category in categories:
        img_folder = os.path.join(images_root, category)
        text_folder = os.path.join(text_root, category)
        if not os.path.isdir(img_folder):
            continue

        img_files = [f for f in os.listdir(img_folder)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if max_per_class:
            img_files = img_files[:max_per_class]

        print(f"Processing {category} ({len(img_files)} images)...")

        for img_file in img_files:
            img_path = os.path.join(img_folder, img_file)

            # --- image features ---
            try:
                img = image.load_img(img_path, target_size=(224, 224))
                x = image.img_to_array(img)
                x = np.expand_dims(x, axis=0)
                x = preprocess_input(x)
                img_feat = model_cnn.predict(x, verbose=0).flatten()
            except Exception as e:
                print(f"  Skipping {img_file} - image load error: {e}")
                continue
            image_features.append(img_feat)

            # --- text features (cleaned OCR) ---
            json_file = img_file.rsplit('.', 1)[0] + '.json'
            json_path = os.path.join(text_folder, json_file)
            if os.path.exists(json_path):
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                raw_tokens = [item['description'] for item in data]
                cleaned_text = clean_ocr_text(raw_tokens)
            else:
                cleaned_text = ""
            text_data.append(cleaned_text)

            labels.append(category)

    return np.array(image_features), text_data, labels

# NOTE: set max_per_class=30 first to do a QUICK test run (~15-20 min given 21 categories).
# Once you confirm everything works, re-run with max_per_class=None for the full dataset.
img_features, text_data, labels = load_multimodal_data(IMAGES_ROOT, TEXT_ROOT, max_per_class=30)
print(f"\nLoaded {len(img_features)} samples total.")


**Note on the quick test run:** `max_per_class=30` verifies the pipeline works end-to-end
before committing to the full ~11,900 image run. Once Cell 8 (training) runs successfully,
come back to Cell 4, change it to `max_per_class=None`, and re-run Cells 4 onward for the
full dataset. Given ~11,900 images, expect this full run to take considerably longer than
Step 1's OCR pass - possibly 1-2+ hours since ResNet50 processes each image individually.
Keep your laptop from sleeping (same as before) and avoid closing the tab.

## Cell 5 - TF-IDF text feature extraction

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=500,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.8
)
text_features = vectorizer.fit_transform(text_data).toarray()
print("Text feature shape:", text_features.shape)


## Cell 6 - Normalize + fuse features

In [ ]:
scaler_img = StandardScaler()
img_features_norm = scaler_img.fit_transform(img_features)

scaler_text = StandardScaler()
text_features_norm = scaler_text.fit_transform(text_features)

fused_features = np.concatenate([img_features_norm, text_features_norm], axis=1)
print("Fused feature shape:", fused_features.shape)

unique_labels = sorted(set(labels))
label_to_int = {label: i for i, label in enumerate(unique_labels)}
labels_int = np.array([label_to_int[l] for l in labels])
labels_onehot = to_categorical(labels_int, num_classes=len(unique_labels))

print(f"Classes ({len(unique_labels)}):", unique_labels)
print("Class distribution:", Counter(labels))


## Cell 7 - Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    fused_features, labels_onehot,
    test_size=0.2,
    random_state=42,
    stratify=labels_int
)
print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")


## Cell 8 - Build and train the classifier

In [ ]:
def create_model(input_dim, num_classes):
    model = Sequential([
        Dense(512, activation='relu', input_shape=(input_dim,), kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.4),
        Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ])
    return model

model = create_model(fused_features.shape[1], len(unique_labels))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")
print(f"Final Test Loss: {test_loss:.4f}")


**Note:** batch size increased to 32 here (vs 16 for the small Freiburg quick-test) since
the full dataset is much larger (~11,900 vs ~5,000 images) - larger batches train faster
and more stably at this scale.

## Cell 9 - Save the model + all preprocessors

In [ ]:
model.save('/content/safescan_classifier.h5')

with open('/content/preprocessors.pkl', 'wb') as f:
    pickle.dump({
        'vectorizer': vectorizer,
        'scaler_img': scaler_img,
        'scaler_text': scaler_text,
        'label_to_int': label_to_int,
        'unique_labels': unique_labels
    }, f)

print("Saved: safescan_classifier.h5 and preprocessors.pkl")


## Cell 10 - Download the saved files (you'll need these for the backend)

In [ ]:
from google.colab import files
files.download('/content/safescan_classifier.h5')
files.download('/content/preprocessors.pkl')


## Next step

Once you're happy with the accuracy on the full dataset run (`max_per_class=None`), you're
ready for **Step 3: Backend (FastAPI)** - this loads `safescan_classifier.h5` and
`preprocessors.pkl`, exposes a `/predict` endpoint, and adds the Gemini LLM layer for
allergen + nutrition analysis. Message Claude to get that set up.
